## module 3e — layered layouts (iteration 4)

13's plans still had blob drifts on bare ground, one plant per stratum, and metrics
that were the generator's own rules restated (circular: real plans scored *low*).
this iteration rebuilds representation + metrics from planting-design literature
before touching the model:

1. **strata, not a flat canvas** — groundcover carpets the bed *under* taller layers
   ("green mulch", Rainer & West, *Planting in a Post-Wild World*); overlap only
   counts within a stratum. layer count shares follow their 10-15% structural /
   25-40% seasonal / ~50% ground bands.
2. **drift geometry** — elongated (aspect >= ~2.5), oblique, interlocking drifts
   (Jekyll, *Colour in the Flower Garden*); theme plant repeated >=3x at
   quasi-regular intervals (Oudolf & Kingsbury, *Planting: A New Perspective*).
3. **an independent metric suite** (`score3`) — form/texture adjacency contrast
   (UF/IFAS ENH1172 fine/medium/coarse), min-over-season interest windows,
   Moon-Spencer-style hue harmony (validated for plant communities, Color Res. &
   Appl. 2025), and per-species clustering at drift scale (Ripley 1977's K, the
   spatial-stats standard). these come from literature, not from `gen_plan3` — the
   anti-circularity fix.
4. **calibration on real plans** — ~330 structured community plans harvested from
   permapeople.org (plant coordinates + species, no pixel extraction); designed
   ornamental beds score 0.67-0.81 on the geometry subset vs 0.47 mean, so the
   metrics rank real design quality sensibly.

palette grows 16 -> 21 (appended, so old checkpoints keep their indices): spire /
plume / screen forms and the aug-oct bloom gap. runtime: ~20 min corpus, ~40 min
training on a small gpu, checkpointed + resumable
(`checkpoints/garden_maskdiff3_{last,best}.pt`).

In [ ]:
# get bvtrain (shared plumbing): locally it's ../bvtrain; on colab/kaggle clone the repo
import os, sys
_CANDS = ["..", ".", "botanical-vision"]
if not any(os.path.isdir(f"{p}/bvtrain") for p in _CANDS):
    os.system("git clone -q https://github.com/babnigg/botanical-vision")
sys.path[:0] = [p for p in _CANDS if os.path.isdir(f"{p}/bvtrain")]

import hashlib, json, math, random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm, trange

from bvtrain import garden as g

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.facecolor"] = "linen"
DEV

### teacher check: is the layered generator actually better?

`gen_plan3` + `repair3` vs 13's `gen_plan2` + `repair`, 100 sites each, on the new
independent suite. the eye-check grid follows — v2 plans show bare wheat ground and
round clusters; v3 shows knitted mats with oblique drifts weaving through.

In [ ]:
def bench(fn, n=100, desc=""):
    rows = []
    for _ in trange(n, desc=desc, leave=False):
        w, d, sun = g.rand_site()
        rows.append(g.score3(fn(w, d, sun), w, d, sun))
    return pd.DataFrame(rows).mean().round(3)

results = pd.DataFrame({
    "rules v2 + repair": bench(lambda w, d, s: g.repair(g.gen_plan2(w, d, s), w, d), desc="v2"),
    "rules v3": bench(g.gen_plan3, desc="v3"),
    "v3 + repair3": bench(lambda w, d, s: g.repair3(g.gen_plan3(w, d, s), w, d), desc="v3r"),
})
results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for c in range(3):
    w, d, sun = g.rand_site()
    g.show_plan(g.repair(g.gen_plan2(w, d, sun), w, d), w, d, axes[0][c],
                f"v2 designed · {w}x{d} {g.SUN_NAMES[sun]}")
    g.show_plan(g.repair3(g.gen_plan3(w, d, sun), w, d), w, d, axes[1][c],
                f"v3 layered · {w}x{d} {g.SUN_NAMES[sun]}")
plt.tight_layout()

### canvas v3

same token scheme as 13 (site prefix + per-species count tokens + (species, x, y)
slots), stretched for the layered corpus: 21 species, up to 110 plants (carpets are
many small plants). count tokens still bucket at 0-9 — saturated for groundcover,
which is fine: you pin "2 hydrangea", not "37 vinca".

In [ ]:
NX, NY, NW, ND, NCNT = 32, 12, 9, 7, 10
NSP = len(g.PALETTE)
MASK  = 0
T_W   = 1
T_D   = T_W + NW
T_SUN = T_D + ND
T_CNT = T_SUN + 3
T_SP  = T_CNT + NCNT                    # slot species: idx 0 = empty
T_X   = T_SP + 1 + NSP                  # idx 0 = none
T_Y   = T_X + 1 + NX                    # idx 0 = none
VOCAB = T_Y + 1 + NY
MAXP  = 110
L     = 3 + NSP + 3 * MAXP

FAM = ([(T_W, T_D), (T_D, T_SUN), (T_SUN, T_CNT)] + [(T_CNT, T_SP)] * NSP
       + [(T_SP, T_X), (T_X, T_Y), (T_Y, VOCAB)] * MAXP)
FAM_MASK = torch.full((L, VOCAB), float("-inf"))
for p, (lo, hi) in enumerate(FAM):
    FAM_MASK[p, lo:hi] = 0
MASKABLE = torch.zeros(L, dtype=torch.bool); MASKABLE[3:] = True

def site_tokens(w, d, sun):
    return [T_W + min(NW - 1, int((w - 3.5) / 0.5)),
            T_D + min(ND - 1, int((d - 1.8) / 0.2)), T_SUN + sun]

def encode(plan, w, d, sun):
    counts = Counter(i for i, _, _, _ in plan)
    t = site_tokens(w, d, sun) + [T_CNT + min(NCNT - 1, counts.get(i, 0)) for i in range(NSP)]
    for i, x, y, r in sorted(plan, key=lambda p: (-p[2], p[1]))[:MAXP]:   # back-to-front
        t += [T_SP + 1 + i, T_X + 1 + min(NX - 1, int(x / w * NX)),
              T_Y + 1 + min(NY - 1, int(y / d * NY))]
    t += [T_SP, T_X, T_Y] * (MAXP - (len(t) - 3 - NSP) // 3)
    return t

def decode(tokens, w, d):
    plan = []
    for k in range(3 + NSP, L, 3):
        s, x, y = tokens[k] - T_SP, tokens[k + 1] - T_X, tokens[k + 2] - T_Y
        if s <= 0 or x <= 0 or y <= 0:
            continue
        i = s - 1
        plan.append((i, (x - 0.5) / NX * w, (y - 0.5) / NY * d, g.PALETTE[i]["s"] / 200))
    return plan

In [ ]:
N_TRAIN, N_VAL = 12000, 500
sites = [g.rand_site() for _ in range(N_TRAIN + N_VAL)]
X = torch.tensor([encode(g.repair3(g.gen_plan3(w, d, s), w, d), w, d, s)
                  for w, d, s in tqdm(sites, desc="layered corpus")])
Xtr, Xval = X[:N_TRAIN], X[N_TRAIN:]
Xtr.shape

In [ ]:
class LayoutMDM(nn.Module):
    def __init__(self, v=VOCAB, dm=192, heads=6, layers=6, ctx=L):
        super().__init__()
        self.tok = nn.Embedding(v, dm); self.pos = nn.Embedding(ctx, dm)
        blk = nn.TransformerEncoderLayer(dm, heads, dm * 4, dropout=0.1,
                                         batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(blk, layers, enable_nested_tensor=False)
        self.head = nn.Linear(dm, v)
    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        return self.head(self.tf(h))

model = LayoutMDM().to(DEV)

EPOCHS = 40
FRESH = False
CFG = dict(n=N_TRAIN, epochs=EPOCHS, dm=192, layers=6, seed=SEED, vocab=VOCAB, maxp=MAXP, v=3)
SIG = hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:10]
LAST, BEST = g.ckpt_dir() / "garden_maskdiff3_last.pt", g.ckpt_dir() / "garden_maskdiff3_best.pt"

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
start_ep, best_val, hist = 0, float("inf"), {"train": [], "val": []}
if LAST.exists() and not FRESH:
    ck = torch.load(LAST, map_location=DEV, weights_only=False)
    if ck.get("sig") == SIG:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start_ep, best_val, hist = ck["epoch"], ck["best_val"], ck["hist"]
        print(f"resumed at epoch {start_ep} (best val {best_val:.3f})")
    else:
        print("config changed — starting fresh")
sum(p.numel() for p in model.parameters())

In [ ]:
def masked_batch(xb, gen=None):
    B = xb.shape[0]
    ratio = torch.cos(torch.rand(B, 1, generator=gen) * math.pi / 2).clamp(min=0.05)
    m = (torch.rand(B, L, generator=gen) < ratio) & MASKABLE
    xin = xb.clone(); xin[m] = MASK
    return xin, m

loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr), batch_size=48, shuffle=True)
vloader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xval), batch_size=128)

for ep in range(start_ep, EPOCHS):
    model.train(); tot = n = 0
    bar = tqdm(loader, desc=f"epoch {ep + 1}/{EPOCHS}", leave=False)
    for (xb,) in bar:
        xin, m = masked_batch(xb)
        loss = F.cross_entropy(model(xin.to(DEV))[m.to(DEV)], xb.to(DEV)[m.to(DEV)])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); n += 1
        bar.set_postfix(loss=f"{tot / n:.3f}")
    model.eval(); vgen = torch.Generator().manual_seed(123); vtot = vn = 0
    with torch.no_grad():
        for (xb,) in vloader:
            xin, m = masked_batch(xb, vgen)
            vtot += F.cross_entropy(model(xin.to(DEV))[m.to(DEV)], xb.to(DEV)[m.to(DEV)]).item()
            vn += 1
    tr, vl = tot / n, vtot / vn
    hist["train"].append(tr); hist["val"].append(vl)
    star = ""
    if vl < best_val:
        best_val, star = vl, " *"
        torch.save({"sig": SIG, "model": model.state_dict(), "val": vl}, BEST)
    torch.save({"sig": SIG, "model": model.state_dict(), "opt": opt.state_dict(),
                "epoch": ep + 1, "best_val": best_val, "hist": hist}, LAST)
    print(f"epoch {ep + 1:2d}  train {tr:.3f}  val {vl:.3f}{star}")

ck = torch.load(BEST, map_location=DEV, weights_only=False)
model.load_state_dict(ck["model"]); model.eval()
print(f"using best (val {ck['val']:.3f})")

### sampling + decode-time search

maskgit unmasking + best-of-N reranked by `score3` + `repair3` — the search now
optimizes the independent suite, so weaknesses the teacher leaves (hue harmony isn't
enforced at generation) get picked up at decode time.

In [ ]:
FAM_DEV = FAM_MASK.to(DEV)

@torch.no_grad()
def sample_plans(plan_sites, steps=12, temp=1.0, count_pins=None):
    B = len(plan_sites)
    canvas = torch.zeros(B, L, dtype=torch.long)
    fixed = torch.zeros(B, L, dtype=torch.bool)
    for b, (w, d, sun) in enumerate(plan_sites):
        canvas[b, :3] = torch.tensor(site_tokens(w, d, sun)); fixed[b, :3] = True
        for sp, cnt in (count_pins[b] if count_pins else {}).items():
            canvas[b, 3 + sp] = T_CNT + min(NCNT - 1, cnt); fixed[b, 3 + sp] = True
    canvas, fixed = canvas.to(DEV), fixed.to(DEV)
    canvas[~fixed] = MASK
    M0 = (canvas == MASK).sum(1)
    for t in range(steps):
        logits = model(canvas) + FAM_DEV
        probs = F.softmax(logits / temp, -1)
        samp = torch.multinomial(probs.reshape(-1, VOCAB), 1).reshape(B, L)
        conf = probs.gather(-1, samp.unsqueeze(-1)).squeeze(-1)
        still = canvas == MASK
        canvas = torch.where(still, samp, canvas)
        n_mask = (M0.float() * math.cos(math.pi / 2 * (t + 1) / steps)).long()
        conf = conf.masked_fill(~still, float("inf"))
        for b in range(B):
            if n_mask[b] > 0:
                canvas[b, conf[b].argsort()[:n_mask[b]]] = MASK
    plans = [decode(canvas[b].tolist(), s[0], s[1]) for b, s in enumerate(plan_sites)]
    cnt_tok = (canvas[:, 3:3 + NSP] - T_CNT).cpu()
    return plans, cnt_tok

def count_obedience(plans, cnt_tok):
    diffs = []
    for plan, toks in zip(plans, cnt_tok):
        placed = Counter(i for i, _, _, _ in plan)
        diffs += [abs(placed.get(i, 0) - min(int(toks[i]), NCNT - 1)) for i in range(NSP)]
    return float(np.mean(diffs))

def best_of(site, n=8, count_pins=None):
    plans, _ = sample_plans([site] * n, count_pins=[count_pins] * n if count_pins else None)
    plans = [g.repair3(p, site[0], site[1]) for p in plans]
    return max(plans, key=lambda p: g.score3(p, *site)["score"])

In [ ]:
eval_sites = [g.rand_site() for _ in range(100)]

plans1, ct1 = sample_plans(eval_sites)
obed = count_obedience(plans1, ct1)

def score_plans(plans):
    return pd.DataFrame([g.score3(p, w, d, s)
                         for p, (w, d, s) in zip(plans, eval_sites)]).mean().round(3)

comp = pd.DataFrame({
    "random": score_plans([g.gen_random(w, d) for w, d, s in eval_sites]),
    "teacher (v3+repair3)": score_plans([g.repair3(g.gen_plan3(w, d, s), w, d)
                                         for w, d, s in tqdm(eval_sites, desc="teacher", leave=False)]),
    "model ×1": score_plans(plans1),
    "model ×1 + repair3": score_plans([g.repair3(p, w, d) for p, (w, d, s) in zip(plans1, eval_sites)]),
    "model best-of-8 + repair3": score_plans([best_of(s) for s in tqdm(eval_sites, desc="best-of-8", leave=False)]),
})
print(f"count-token obedience: mean |placed - token| = {obed:.2f} plants/species")
comp

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for c in range(3):
    site = g.rand_site()
    p1, _ = sample_plans([site])
    g.show_plan(p1[0], site[0], site[1], axes[0][c], f"×1 sample · {site[0]}x{site[1]}")
    g.show_plan(best_of(site), site[0], site[1], axes[1][c], "best-of-8 + repair3")
plt.tight_layout()

### how do real plans compare now?

the same geometry-only subset (no bloom/color data in the harvested records) on:
top permapeople community beds, the v3 teacher, and the model. in 13/14 the metrics
couldn't *see* what real plans did well; the calibrated suite ranks designed beds
0.67-0.81 — and the model should sit near the teacher, not above the best humans.

In [ ]:
import importlib.util as ilu
spec = ilu.spec_from_file_location("pp", os.path.join("..", "scripts", "permapeople_plans.py"))
pp = ilu.module_from_spec(spec); spec.loader.exec_module(pp)

def geo_subset(plan, w, d):
    return {k: v for k, v in g.score3(plan, w, d).items()
            if k in ("overlap", "ground", "layers", "drift", "rhythm", "cluster")}

real = []
if pp.RAW.exists():
    for f in sorted(pp.RAW.glob("*.json")):
        if f.name == "manifest.json":
            continue
        try:
            out = pp.load_plan(f)
        except (KeyError, ValueError, TypeError):
            continue
        if out and out[4]["n"] >= 15 and out[3] <= 12 and out[2] <= 30:   # bed-scale only
            m = pp.score_real(*out[:4])
            real.append(np.mean(list({k: m[k] for k in ("overlap", "ground", "layers",
                                                        "drift", "rhythm", "cluster")}.values())))
rows = {
    "real beds (permapeople)": (np.mean(sorted(real, reverse=True)[:20]) if real else float("nan")),
    "teacher v3": np.mean([np.mean(list(geo_subset(g.repair3(g.gen_plan3(w, d, s), w, d), w, d).values()))
                           for w, d, s in eval_sites[:30]]),
    "model best-of-8": np.mean([np.mean(list(geo_subset(best_of(s), s[0], s[1]).values()))
                                for s in tqdm(eval_sites[:30], desc="model", leave=False)]),
}
print(f"{len(real)} real bed-scale plans scored (top-20 mean reported)")
pd.Series(rows, name="geometry-subset mean").round(3)

### toolbox completion via count tokens

same interaction as 13 — "2 hydrangea + 7 coneflower" is two pinned count tokens —
now completed inside a fully-planted layered bed.

In [ ]:
HYD = next(i for i, p in enumerate(g.PALETTE) if "Hydrangea" in p["name"])
ECH = next(i for i, p in enumerate(g.PALETTE) if "Echinacea" in p["name"])
site = (5.5, 2.6, 2)
pins = {HYD: 2, ECH: 7}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
briefs = []
for k, ax in enumerate(axes):
    plan = best_of(site, count_pins=pins)
    got = Counter(i for i, _, _, _ in plan)
    briefs.append((got.get(HYD, 0), got.get(ECH, 0)))
    hl = [j for j, (i, *_) in enumerate(plan) if i in (HYD, ECH)]
    g.show_plan(plan, site[0], site[1], ax, f"completion {k + 1}: 2 hydrangea + 7 coneflower", pins=hl)
plt.tight_layout()
briefs

### takeaways

- (numbers filled in after the run)